In [1]:
import numpy as np
import pandas as pd
import os, glob, re, math, shutil
import scipy
from Bio import Seq, SeqIO
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import hvplot.pandas
import matplotlib
import matplotlib.dates as mdates
from pathlib import Path
from matplotlib.patches import Rectangle
from shapely import wkt
from geopy.distance import geodesic

/Users/szk017/miniforge3/lib/python3.13/site-packages/Bio/__init__.py:138: BiopythonWarning: You may be importing Biopython from inside the source tree. This is bad practice and might lead to downstream issues. In particular, you might encounter ImportErrors due to missing compiled C extensions. We recommend that you try running your code from outside the source tree. If you are outside the source tree then you have a pyproject.toml file in an unexpected directory: /Users/szk017/miniforge3/lib/python3.13/site-packages
  warnings.warn(


In [2]:
lineage = 'III'
df_seq = pd.DataFrame([(seq.id, seq.seq) for seq in SeqIO.parse(f"phylogeny/Lineage_{lineage}_core_genome.aln", "fasta")], columns=['Sample', 'Seq'])
len(df_seq), len(df_seq.Seq.values[0])

(13, 235088)

In [3]:
df_seq = pd.DataFrame([(seq.id, seq.seq) for seq in SeqIO.parse(f"phylogeny/Lineage_{lineage}/core_genome_SNPs.aln", "fasta")], columns=['Sample', 'Seq'])
len(df_seq), len(df_seq.Seq.values[0])

(13, 83194)

In [4]:
df_collection_metadata = pd.read_csv("./data/collection_metadata.csv")
df_PN_metadata = pd.read_csv("data/from_Reed/101526_pnusa.csv")[['county_name', 'sample_type', 'Sequence ID']].rename(columns={'county_name': 'County', 'Sequence ID': 'acc_id'})

df_cgmlst1748 = pd.read_excel("./references/BIGSdb_1903258_9055465447_99637.xlsx", sheet_name='output').dropna(axis=0, how='all')
len(df_cgmlst1748)

1748

In [5]:
df_metadata = []

for sample in os.listdir("results"):

    df_mlst = pd.read_csv(f"results/{sample}/strain_typing/mlst.csv")
    df_lineage = pd.read_csv(f"results/{sample}/strain_typing/lineage_CC.csv")

    # prioritize the results from the Pasteur table because the mlst tool results include the ~, meaning some won't get typed
    # ~N means it's the same length as that allele and by default ≥95% identity. I think that's close enough, so we're considering it the same
    del df_mlst['ST']
    df_mlst = pd.concat([df_mlst, df_lineage], axis=1)
    df_mlst['Sample'] = sample

    # skip the first column, which is the file name
    df_metadata.append(df_mlst.iloc[:, 1:])

df_metadata = pd.concat(df_metadata)

df_split_alleles = df_metadata['ALLELES'].str.split(';', expand=True)
df_split_alleles.columns = [f'MLST{num+1}' for num in range(df_split_alleles.shape[1])]

df_metadata = pd.concat([df_metadata, df_split_alleles], axis=1)
df_metadata['acc_id'] = df_metadata['Sample'].str.split('-').str[0]

combine_cols = ['acc_id', 'ST', 'CC', 'Lineage', 'MLST1', 'MLST2', 'MLST3', 'MLST4', 'MLST5', 'MLST6', 'MLST7']

df_metadata = df_metadata.merge(pd.concat([df_collection_metadata, df_PN_metadata]), how='left')
# df_metadata["geometry"] = df_metadata["geometry"].apply(wkt.loads)

df_metadata['Collection date'] = pd.to_datetime(df_metadata['Collection date'])

df_metadata['WWTP_Cases'] = df_metadata.groupby('WWTP')['Sample'].transform(lambda x: x.count())

print(f"{len(df_metadata.loc[pd.isnull(df_metadata['Lineage'])])}/{len(df_metadata)} samples don't have lineage and CC typing")


20/366 samples don't have lineage and CC typing


In [6]:
# df_blast_results = []

# for sample in os.listdir("results"):

#     if sample.startswith('PN'):
        
#         df_blast = pd.read_csv(f"./results/{sample}/strain_typing/cgMLST1748_blast_results.tsv.gz", sep='\t', 
#                                names=['qseqid', 'sseqid', 'pident', 'length', 'mismatch', 'gapopen', 'qstart', 'qend', 'sstart', 'send', 'evalue', 'bitscore']
#                                )
    
#         df_blast['Sample'] = sample
#         df_blast_results.append(df_blast)

# df_blast_results = pd.concat(df_blast_results)
# print(len(df_blast_results))

# # add information about the locus lengths so that we can keep only hits that cover at least 90% of the average locus length
# # df_blast_results['s_aln_len'] = np.abs(df_blast_results['send'] - df_blast_results['sstart']) + 1
# df_blast_results['locus'] = df_blast_results['sseqid'].str.split('_').str[0]
# df_blast_results['allele'] = df_blast_results['sseqid'].str.split('_').str[1].astype(int)

In [96]:
df_allele_typing = df_blast_results.sort_values(['Sample', 'locus', 'bitscore'], ascending=[True, True, False]).drop_duplicates(['Sample', 'locus'], keep='first')
df_allele_typing['allele'] = df_allele_typing['allele'].astype(str)
df_allele_typing.set_index(['Sample']).to_csv("./data/cgmlst1758_allele_typing_clinical.csv")

In [98]:
df_allele_typing.Sample.nunique()

235

In [99]:
df_allele_typing['s_aln_len'] = np.abs(df_allele_typing['send'] - df_allele_typing['sstart']) + 1
df_allele_typing = df_allele_typing.merge(df_cgmlst1748[['locus', 'min length']], how='left')
df_allele_typing['locus_prop_length'] = df_allele_typing['s_aln_len'] / df_allele_typing['min length']

In [100]:
df_allele_typing.locus_prop_length.min()

np.float64(0.058333333333333334)

In [101]:
df_cgmlst1748.query("locus=='lmo2118'")

,scheme,locus,data type,alleles,length varies,standard length,min length (setting),max length (setting),min length,max_length,full name/product,aliases,curators
1295,cgMLST1748,lmo2118,DNA,1099.0,True,1353.0,NaN,NaN,1353.0,1356.0,phosphoglucosamine mutase,NaN,NaN


In [105]:
df_loci_by_sample = pd.DataFrame(df_allele_typing.groupby('Sample').locus.nunique()).reset_index()

In [106]:
len(df_loci_by_sample.query("locus != 1748")), df_loci_by_sample.query("locus != 1748").locus.min()

(44, np.int64(1742))

In [112]:
df_loci_by_sample.query("locus < 1747")

,Sample,locus
137,PNUSAL023663,1745
162,PNUSAL024917,1742


In [107]:
df_allele_typing.evalue.max()

np.float64(0.27)

In [108]:
df_missing_loci_after_blast = []

for sample in df_loci_by_sample.query("locus != 1748").Sample.unique():

    missing_loci = list(set(df_allele_typing.locus) - set(df_allele_typing.query("Sample==@sample").locus))
    df_add = df_cgmlst1748.query("locus in @missing_loci")
    df_add['Sample'] = sample

    df_missing_loci_after_blast.append(df_add)

df_missing_loci_after_blast = pd.concat(df_missing_loci_after_blast)
print(len(df_missing_loci_after_blast))

51


In [109]:
df_missing_loci_after_blast['full name/product'].value_counts()

full name/product
hypothetical protein                     45
GntR family transcriptional regulator     1
PTS sorbitol transporter subunit IIBC     1
PTS sorbitol transporter subunit IIC      1
NAD(P)-dependent oxidoreductase           1
DeoR family transcriptional regulator     1
bile acid hydrolase                       1
Name: count, dtype: int64

In [110]:
df_missing_loci_after_blast['locus'].value_counts()

locus
lmo2707    32
lmo0510    10
lmo0741     1
lmo0743     1
lmo0543     1
lmo0544     1
lmo0545     1
lmo0546     1
lmo0547     1
lmo1126     1
lmo2067     1
Name: count, dtype: int64

In [496]:
# df_missing_loci_after_blast[['Sample', 'locus']].sort_values('Sample')

In [63]:
# this locus is missing from all the CC554 and CC389 samples. There are no CC554 and CC389 samples with it, cool!
# it's not a length thing, like there are matches but they are lower quality or got filtered out by 90% length filter. They aren't returned at all with a 70% minimum
df_missing_loci_after_blast.query("locus=='lmo2707'")[['Sample']].merge(df_metadata)[['Lineage', 'CC']].value_counts(dropna=False)

Lineage  CC   
I        CC554    26
         CC389    12
NaN      NaN       1
Name: count, dtype: int64

In [64]:
df_metadata.query("CC in ['CC554', 'CC389']").shape

(38, 30)

In [66]:
df_missing_loci_after_blast.query("locus=='lmo2707'")[['Sample']].merge(df_metadata).query("Lineage != 'I'")

,Sample,SCHEME,STATUS,SCORE,ALLELES,abcZ,bglA,cat,dapE,dat,...,MLST6,MLST7,acc_id,Collection date,WWTP,County,wwtp_latitude,wwtp_longitude,geometry,WWTP_Cases
32,EHS2600021507-01-00,listeria_2,NOVEL,90,abcZ(7);bglA(6);cat(10);dapE(8);dat(1);ldh(37)...,7,6,10,8,1,...,ldh(37),lhkA(1),EHS2600021507,2026-09-01,Saratoga Co SD No 1,Saratoga,42.88,-73.69,POLYGON ((-73.69704622976793 43.11105929505071...,11


In [68]:
# couldn't find these even after increasing to 10 million matches, so maybe they don't exist in these assemblies, or there's an issue with my assemblies
# like they're at contig boundaries or something
# could take the FASTAs of the alleles and align reads and see what happens...
df_missing_loci_after_blast[['Sample', 'locus', 'full name/product']].query("locus != 'lmo2707'").merge(df_metadata).sort_values('CC')[['Sample', 'acc_id', 'locus', 'full name/product', 'ST', 'CC', 'Lineage']]

,Sample,acc_id,locus,full name/product,ST,CC,Lineage
0,EHS2500012533-04-01,EHS2500012533,lmo1126,hypothetical protein,29.0,CC29,II
2,EHS2500082385-01-01,EHS2500082385,lmo2067,bile acid hydrolase,631.0,CC4,I
3,EHS2500089134-01-02,EHS2500089134,lmo0510,hypothetical protein,669.0,CC669,III
1,EHS2500055014-01-00,EHS2500055014,lmo0510,hypothetical protein,NaN,NaN,NaN
4,EHS2600010400-01-00,EHS2600010400,lmo0510,hypothetical protein,NaN,NaN,NaN


# Make separate `core_genome.aln` files for each lineage

For samples without a lineage, put them in the group to which they have the smallest cgmlst1748 allelic distance to.

In [7]:
def convert_matrix_to_long(matrix, col_name, df_metadata):

    matrix.index.name = 'Sample1'
    matrix.columns.name = 'Sample2'
    
    df_long = matrix.where(np.triu(np.ones(matrix.shape)).astype(np.bool)).stack().reset_index().dropna().rename(columns={0: col_name})
    df_long[col_name] = df_long[col_name].astype(int)
    
    df_long['acc_id1'] = df_long['Sample1'].str.split('-').str[0]
    df_long['acc_id2'] = df_long['Sample2'].str.split('-').str[0]
    
    # df_long = df_long.query(f"~((Sample1 == Sample2) | (acc_id1 == acc_id2 & {col_name} == 0))").reset_index(drop=True)
    df_long = df_long.query("Sample1 != Sample2").reset_index(drop=True)

    metadata_cols = ['Sample', 'CC', 'Lineage', 'Collection date', 'WWTP', 'County', 'wwtp_latitude', 'wwtp_longitude', 'geometry']
    df_long = df_long.merge(df_metadata[metadata_cols].rename(columns={col: f"{col}1" for col in metadata_cols})).merge(df_metadata[metadata_cols].rename(columns={col: f"{col}2" for col in metadata_cols}))
    
    df_long['Same_CC'] = (df_long['CC1']==df_long['CC2']).astype(int)
    df_long['Same_Lineage'] = (df_long['Lineage1']==df_long['Lineage2']).astype(int)
    
    df_long.loc[pd.isnull(df_long['Lineage1']) | pd.isnull(df_long['Lineage2']), 'Same_Lineage'] = np.nan
    df_long.loc[pd.isnull(df_long['CC1']) | pd.isnull(df_long['CC2']), 'Same_CC'] = np.nan
    
    df_long.loc[(df_long['Same_CC']==0) & (df_long['Same_Lineage']==1), 'Group'] = 'Same Lineage Only'
    df_long.loc[(df_long['Same_CC']==1) & (df_long['Same_Lineage']==1), 'Group'] = 'Same CC and Lineage'
    df_long.loc[(df_long['Same_CC']==0) & (df_long['Same_Lineage']==0), 'Group'] = 'Different CC and Lineage'

    df_long['Collection_date_diff_days'] = np.abs((df_long['Collection date1'] - df_long['Collection date2']).dt.days)
    
    coord_cols = [
        "wwtp_latitude1", "wwtp_longitude1",
        "wwtp_latitude2", "wwtp_longitude2"
    ]
    
    df_long["WWTP_dist_miles"] = df_long.apply(
        lambda row: geodesic(
            (row["wwtp_latitude1"], row["wwtp_longitude1"]),
            (row["wwtp_latitude2"], row["wwtp_longitude2"])
        ).miles
        if row[coord_cols].notna().all()
        else np.nan,
        axis=1
    )

    return df_long

In [8]:
df_allele_typing = pd.concat([pd.read_csv(fName) for fName in glob.glob("data/cgmlst1758_allele_typing*.csv")])

cgmlst1748_allele_matrix = df_allele_typing.pivot(index='Sample', columns='locus', values='allele')
cgmlst1748_allele_matrix.to_csv("./data/cgmlst1748_matrix.csv")

# Calculate pairwise allelic distances
allele_dist_matrix = pd.DataFrame(
    scipy.spatial.distance.squareform(scipy.spatial.distance.pdist(cgmlst1748_allele_matrix.to_numpy(dtype=object), metric=lambda a, b: np.sum(a != b))),
    index=cgmlst1748_allele_matrix.index,
    columns=cgmlst1748_allele_matrix.index
).astype(int)

df_allele_dist = convert_matrix_to_long(allele_dist_matrix, 'Allele_Diff', df_metadata)


In [9]:
samples_no_lineage = df_metadata.loc[pd.isnull(df_metadata['Lineage'])].Sample.values
len(samples_no_lineage)

20

In [10]:
samples_inferred_lineage = {}

for sample in samples_no_lineage:

    df_sorted_dist = pd.DataFrame(allele_dist_matrix.loc[sample].sort_values()).reset_index().query("Sample2 != @sample").rename(columns={'Sample2': 'Sample'})
    df_sorted_dist = df_sorted_dist.merge(df_metadata[['Sample', 'Lineage']]).dropna(subset='Lineage')
    samples_inferred_lineage[sample] = df_sorted_dist.Lineage.values[0]

samples_inferred_lineage

{'PNUSAL025271': 'III',
 'PNUSAL026439': 'III',
 'IDR2400055889-04-03': 'II',
 'EHS2600010400-01-00': 'III',
 'PNUSAL024917': 'III',
 'PNUSAL020033': 'II',
 'PNUSAL025548': 'II',
 'PNUSAL025547': 'III',
 'EHS2500055014-01-00': 'III',
 'EHS2500089139-01-00': 'II',
 'PNUSAL026035': 'II',
 'PNUSAL026469': 'III',
 'PNUSAL012418': 'III',
 'PNUSAL014749': 'II',
 'EHS2600021507-01-00': 'II',
 'PNUSAL026356': 'III',
 'EHS2500002739-01-01': 'II',
 'PNUSAL026357': 'III',
 'PNUSAL022759': 'III',
 'PNUSAL026100': 'II'}

In [13]:
common_string = """
DATASET_COLORSTRIP
#In colored strip datasets, each ID is associated to a color box/strip and can have an optional label. Color can be specified in hexadecimal, RGB or RGBA notation. When using RGB or RGBA notation, you cannot use COMMA as the dataset separator

#lines starting with a hash are comments and ignored during parsing

#=================================================================#
#                    MANDATORY SETTINGS                           #
#=================================================================#
#select the separator which is used to delimit the data below (TAB,SPACE or COMMA).This separator must be used throughout this file.

SEPARATOR TAB
#SEPARATOR COMMA
#SEPARATOR SPACE

#label is used in the legend table (can be changed later)
DATASET_LABEL	Clonal Complex

#dataset color (can be changed later)
COLOR	#ff0000

#=================================================================#
#                    OPTIONAL SETTINGS                            #
#=================================================================#

#If COLOR_BRANCHES is set to 1, branches of the tree will be colored according to the colors of the strips above the leaves.
#When all children of a node have the same color, it will be colored the same, ie. the color will propagate inwards towards the root.
COLOR_BRANCHES	0


#=================================================================#
#     all other optional settings can be set or changed later     #
#           in the web interface (under 'Datasets' tab)           #
#=================================================================#

#Each dataset can have a legend, which is defined using LEGEND_XXX fields below
#For each row in the legend, there should be one shape, color and label.
#Optionally, you can define an exact legend position using LEGEND_POSITION_X and LEGEND_POSITION_Y. To use automatic legend positioning, do NOT define these values
#Optionally, shape scaling can be present (LEGEND_SHAPE_SCALES). For each shape, you can define a scaling factor between 0 and 1.
#To order legend entries horizontally instead of vertically, set LEGEND_HORIZONTAL to 1
#Shape should be a number between 1 and 12, or any protein domain shape definition.
#1: square
#2: circle
#3: star
#4: right pointing triangle
#5: left pointing triangle
#6: checkmark
#7: solid line
#8: dashed line
#9: dotted line
#10:arrow left
#11:arrow right
#12:bidirectional arrow

#width of the colored strip
#STRIP_WIDTH 25

#left margin, used to increase/decrease the spacing to the next dataset. Can be negative, causing datasets to overlap.
#MARGIN 0

#border width; if set above 0, a border of specified width (in pixels) will be drawn around the color strip 
BORDER_WIDTH	1

#border color; used when BORDER_WIDTH is above 0
BORDER_COLOR	#000000

#if set to 1, border will be drawn completely around each colored strip box
COMPLETE_BORDER	1

#always show internal values; if set, values associated to internal nodes will be displayed even if these nodes are not collapsed. It could cause overlapping in the dataset display.
#SHOW_INTERNAL 0


#display or hide the individual label inside each colored strip (when defined in the data below)
#SHOW_STRIP_LABELS 1

#position of the strip label within the box; 'top', 'center' or 'bottom'
#STRIP_LABEL_POSITION center

#strip label size factor (relative to the tree leaf labels)
#STRIP_LABEL_SIZE_FACTOR 1


#rotation of the strip labels; used only in rectangular tree display mode
#STRIP_LABEL_ROTATION 0

#strip label shift in pixels (positive or negative)
#STRIP_LABEL_SHIFT 0

#STRIP_LABEL_COLOR #000000

#draw a black outline around the text (width in pixels)
#STRIP_LABEL_OUTLINE 0.5

#calculate the label color automatically (black or white), based on the darkness of the color strip
#STRIP_LABEL_AUTO_COLOR 0

#display or hide the dataset label above the colored strip
#SHOW_LABELS 1

#dataset label size factor
#SIZE_FACTOR 1

#dataset label rotation
#LABEL_ROTATION 0

#dataset label shift in pixels (positive or negative)
#LABEL_SHIFT 0

#align the dataset label to the tree circle; only applies in circular display mode
#LABEL_ALIGN_TO_TREE,0

#Internal tree nodes can be specified using IDs directly, or using the 'last common ancestor' method described in iTOL help pages

#=================================================================#
#       Actual data follows after the "DATA" keyword              #
#=================================================================#
DATA

#Examples:
#assign a red colored strip to leaf 9606, with label 'Human'
#9606 #ff0000 Human

#assign a green, semi-transparent (alpha 0.5) strip to an internal node, without any label. If 'Show internal values' is set to 'No', this will only be displayed if the node is collapsed. 
#9606|5664 rgba(0,255,0,0.5)
"""

In [14]:
min_samples_CC = 2

core_genome = pd.DataFrame([(seq.id, seq.seq) for seq in SeqIO.parse("./phylogeny/core_genome.aln", "fasta")], columns = ['Sample', 'Seq'])

for lineage in np.sort(df_metadata.dropna(subset='Lineage').Lineage.unique()):

    df_single_lineage_metadata = df_metadata.query("Lineage==@lineage")
    single_lineage_samples = list(df_metadata.query("Lineage==@lineage").Sample.values)

    for sample in samples_no_lineage:
        if samples_inferred_lineage[sample] == lineage:
            single_lineage_samples.append(sample)

    # print(lineage, len(single_lineage_samples))

    single_lineage_core_genome = core_genome.query("Sample in @single_lineage_samples")

    with open(f"./phylogeny/Lineage_{lineage}_core_genome.aln", "w+") as file:
        for _, row in single_lineage_core_genome.iterrows():
            file.write(f">{row['Sample']}\n")
            file.write(f"{str(row['Seq'])}\n")

    with open(f"./phylogeny/Lineage_{lineage}/samples_CC_colorstrip.txt", "w+") as file:
        single_lineage_core_genome = single_lineage_core_genome.merge(df_metadata[['Sample', 'CC']])
        # print(single_lineage_core_genome.CC.nunique())

        if lineage == 'III':
            CCs_to_color = pd.DataFrame(df_single_lineage_metadata['CC'].value_counts()).query("count >= 1").reset_index().sort_values(['count', 'CC'], ascending=[False, True]).CC.values
            singleton_CCs = []
        else:
            CCs_to_color = pd.DataFrame(df_single_lineage_metadata['CC'].value_counts()).query("count >= 2").reset_index().sort_values(['count', 'CC'], ascending=[False, True]).CC.values
            singleton_CCs = pd.DataFrame(df_single_lineage_metadata['CC'].value_counts()).query("count == 1").index.values
            
        print(len(CCs_to_color))
        color_dict = dict(zip(CCs_to_color, sns.color_palette("tab20").as_hex()))

        # assign singletons to black and no CC to white, to be consistent with the minimum spanning tree coloring
        for cc in singleton_CCs:
            color_dict[cc] = '#000000'

        # fill the empty ones with white, which are no CC assigned
        single_lineage_core_genome['color'] = single_lineage_core_genome['CC'].map(color_dict).fillna('#ffffff')

        assert len(single_lineage_core_genome.loc[pd.isnull(single_lineage_core_genome['CC'])].query("color != '#ffffff'")) == 0
        assert len(single_lineage_core_genome.loc[~pd.isnull(single_lineage_core_genome['CC'])].query("color == '#ffffff'")) == 0

        file.write(common_string + "\n\n")
        
        for _, row in single_lineage_core_genome.dropna(subset='color').iterrows():
            file.write(f"{row['Sample']}\t{row['color']}\n")

18
17
2


In [77]:
df_metadata.query("Sample=='PNUSAL025241'")[['CC']]

,CC
69,CC388


In [147]:
df_metadata.query("Sample=='PNUSAL013340'")[['CC']]

,CC
64,CC5


In [155]:
df_metadata.query("Sample in ['PNUSAL013690', 'PNUSAL013340']")[['Sample', 'CC']]

,Sample,CC
64,PNUSAL013340,CC5
179,PNUSAL013690,CC5


In [157]:
df_metadata.query("CC in ['CC5']")[['acc_id', 'abcZ', 'bglA', 'cat', 'dapE', 'dat', 'ldh', 'lhkA']].sort_values('acc_id')

,acc_id,abcZ,bglA,cat,dapE,dat,ldh,lhkA
64,PNUSAL013340,2,1,57,3,3,1,2
179,PNUSAL013690,34,1,57,14,2,1,2
56,PNUSAL013923,2,1,11,3,3,1,7
345,PNUSAL018460,2,1,11,3,3,1,7
333,PNUSAL024604,2,1,11,3,3,1,7
123,PNUSAL024655,2,1,11,3,3,1,7
282,PNUSAL025006,2,1,11,3,3,1,7
26,PNUSAL025191,2,1,11,3,3,1,7
175,PNUSAL025592,2,1,11,3,3,1,7
28,PNUSAL026348,2,1,11,3,3,1,7


In [159]:
df_metadata.query("acc_id=='EHS2600009867'")[['acc_id', 'abcZ', 'bglA', 'cat', 'dapE', 'dat', 'ldh', 'lhkA']].sort_values('acc_id')

,acc_id,abcZ,bglA,cat,dapE,dat,ldh,lhkA
151,EHS2600009867,11,12,4,341,3,1,2


In [81]:
mlst_profiles = pd.read_csv("references/mlst_profiles.tsv", sep='\t')

In [115]:
cc_388_samples = df_metadata.query("CC=='CC388'").Sample.values
cc_389_samples = df_metadata.query("CC=='CC389'").Sample.values

df_metadata.query("CC=='CC388'")[['Sample', 'CC', 'WWTP', 'County']].sort_values('County')

,Sample,CC,WWTP,County
268,PNUSAL018773,CC388,NaN,Albany
17,PNUSAL022544,CC388,NaN,Cayuga
271,PNUSAL025097,CC388,NaN,Cortland
87,PNUSAL014787,CC388,NaN,Dutchess
248,PNUSAL022468,CC388,NaN,Dutchess
22,PNUSAL022349,CC388,NaN,Erie
342,PNUSAL026350,CC388,NaN,Fulton
213,PNUSAL022548,CC388,NaN,Greene
300,PNUSAL022021,CC388,NaN,Kings
207,PNUSAL024025,CC388,NaN,Nassau


In [121]:
df_metadata.query("Sample in @cc_389_samples")[['acc_id', 'abcZ', 'bglA', 'cat', 'dapE', 'dat', 'ldh', 'lhkA']]

,acc_id,abcZ,bglA,cat,dapE,dat,ldh,lhkA
5,EHS2500055583,52,1,12,71,2,1,5
12,EHS2500055013,52,1,12,71,2,1,5
29,PNUSAL022347,52,1,12,71,2,1,5
59,EHS2600002591,52,1,12,71,2,1,5
116,EHS2500081979,52,1,12,71,2,1,5
117,PNUSAL015067,52,1,12,71,2,1,5
127,PNUSAL017483,52,1,12,71,2,1,5
139,EHS2600016313,52,1,12,71,2,1,5
157,EHS2500069203,52,1,12,71,2,1,5
182,EHS2500069202,52,1,12,71,2,1,5


In [111]:
df_metadata.query("acc_id=='PNUSAL025241'")[['acc_id', 'abcZ', 'bglA', 'cat', 'dapE', 'dat', 'ldh', 'lhkA']]

,acc_id,abcZ,bglA,cat,dapE,dat,ldh,lhkA
69,PNUSAL025241,11,1,12,38,2,1,5


In [119]:
df_metadata.query("Sample in @cc_388_samples")[['acc_id', 'abcZ', 'bglA', 'cat', 'dapE', 'dat', 'ldh', 'lhkA', 'CC']]

,acc_id,abcZ,bglA,cat,dapE,dat,ldh,lhkA,CC
17,PNUSAL022544,3,1,12,70,2,1,5,CC388
22,PNUSAL022349,3,102,12,70,2,1,5,CC388
31,PNUSAL022545,3,1,12,70,2,1,5,CC388
51,PNUSAL025445,3,1,12,70,2,1,5,CC388
69,PNUSAL025241,11,1,12,38,2,1,5,CC388
87,PNUSAL014787,3,1,12,70,2,1,5,CC388
132,PNUSAL022560,3,1,12,70,2,1,5,CC388
133,PNUSAL014208,3,1,12,70,2,1,5,CC388
137,PNUSAL022157,3,1,12,70,2,1,5,CC388
143,PNUSAL014636,3,1,12,70,2,1,5,CC388
